In [44]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

In [45]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve,
                             precision_recall_curve, average_precision_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import wilcoxon, friedmanchisquare

In [ ]:
# ── CatBoost 
from catboost import CatBoostClassifier

In [ ]:
# ── Optuna 
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ── SHAP 
import shap

In [ ]:
#  CONFIG

RANDOM_STATE = 42
TEST_SIZE    = 0.20
N_TRIALS     = 150
CV_FOLDS     = 5
FIGURES_DIR  = "figures"
MODELS_DIR   = "models"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
np.random.seed(RANDOM_STATE)

print("=" * 65)
print("  CANCER DU SEIN — CatBoost + Optuna + SHAP | WDBC sklearn")
print("=" * 65)



  CANCER DU SEIN — CatBoost + Optuna + SHAP | WDBC sklearn


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ÉTAPE 1 — CHARGEMENT
# ══════════════════════════════════════════════════════════════════════════════
print("\n[1] Chargement via load_breast_cancer()...")

raw  = load_breast_cancer()
df   = pd.DataFrame(raw.data, columns=raw.feature_names)


df['diagnosis'] = 1 - raw.target  

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

print(f"  Shape      : {df.shape}")
print(f"  Malin  (1) : {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")
print(f"  Bénin  (0) : {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Features   : {list(X.columns[:4])} ...")
print(f"  Manquants  : {df.isnull().sum().sum()}")





[1] Chargement via load_breast_cancer()...
  Shape      : (569, 31)
  Malin  (1) : 212 (37.3%)
  Bénin  (0) : 357 (62.7%)
  Features   : ['mean radius', 'mean texture', 'mean perimeter', 'mean area'] ...
  Manquants  : 0


In [ ]:

#  ÉTAPE 2 — EDA
print("\n[2] EDA...")

# ── Figure 1 : Distribution des classes 
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
counts = y.value_counts().sort_index()
labels = ['Bénin (0)', 'Malin (1)']
colors = ['#2ECC71', '#E74C3C']

bars = axes[0].bar(labels, [counts[0], counts[1]], color=colors,
                   edgecolor='white', linewidth=1.5, width=0.45)
axes[0].set_title('Distribution des Classes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre de patients')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, cnt in zip(bars, [counts[0], counts[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 4,
                 f'{cnt}\n({cnt/len(y)*100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')

axes[1].pie([counts[0], counts[1]], labels=['Bénin', 'Malin'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11})
axes[1].set_title('Répartition Bénin / Malin', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig1_classes.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig1_classes.png")



[2] EDA...
  ✓ fig1_classes.png


In [ ]:
# ── Figure 2 : Heatmap corrélation 
plt.figure(figsize=(16, 13))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=0.3, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
plt.title('Matrice de Corrélation — 30 Features WDBC',
          fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig2_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig2_correlation.png")

  ✓ fig2_correlation.png


In [ ]:
# ── Figure 3 : Boxplots top features 
top8 = ['mean radius', 'mean perimeter', 'mean area', 'mean concavity',
        'worst radius', 'worst perimeter', 'worst concave points', 'worst area']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
df_plot = df.copy()
df_plot['Classe'] = df_plot['diagnosis'].map({1: 'Malin', 0: 'Bénin'})

for i, feat in enumerate(top8):
    if feat in df_plot.columns:
        benin = df_plot[df_plot['Classe']=='Bénin'][feat]
        malin = df_plot[df_plot['Classe']=='Malin'][feat]
        bp = axes[i].boxplot([benin, malin], labels=['Bénin','Malin'],
                             patch_artist=True,
                             medianprops=dict(color='black', linewidth=2))
        bp['boxes'][0].set_facecolor('#2ECC7199')
        bp['boxes'][1].set_facecolor('#E74C3C99')
        axes[i].set_title(feat, fontsize=9, fontweight='bold')
        axes[i].grid(True, alpha=0.3)

plt.suptitle('Top 8 Features — Distribution Bénin vs Malin',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig3_boxplots.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig3_boxplots.png")




  ✓ fig3_boxplots.png


In [ ]:
#  ÉTAPE 3 — PRÉTRAITEMENT

print("\n[3] Prétraitement...")

# Split 80/20 stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"  Train : {X_train.shape}  |  Test : {X_test.shape}")

# SMOTE si déséquilibre > 1.5
ratio = y_train.value_counts()[0] / y_train.value_counts()[1]
if ratio > 1.5:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print(f"  SMOTE appliqué → {dict(pd.Series(y_train_bal).value_counts())}")
else:
    X_train_bal, y_train_bal = X_train.copy(), y_train.copy()
    print(f"  Pas de SMOTE (ratio={ratio:.2f})")

# StandardScaler pour KNN / SVM / LR (fit sur train uniquement)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_test_sc  = scaler.transform(X_test)
print("  StandardScaler ajusté sur train (KNN/SVM/LR)")



[3] Prétraitement...
  Train : (455, 30)  |  Test : (114, 30)
  SMOTE appliqué → {1: np.int64(285), 0: np.int64(285)}
  StandardScaler ajusté sur train (KNN/SVM/LR)


In [ ]:

#  ÉTAPE 4 — BASELINES

print("\n[4] Baselines...")

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

baselines = {
    "KNN"                : KNeighborsClassifier(n_neighbors=5),
    "SVM"                : SVC(kernel='rbf', probability=True,
                               random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(max_iter=10000,
                                              random_state=RANDOM_STATE),
    "Random Forest"      : RandomForestClassifier(n_estimators=100,
                                                  random_state=RANDOM_STATE),
}

results   = {}
cv_scores = {}

for name, model in baselines.items():
    Xtr = X_train_sc if name != "Random Forest" else X_train_bal.values
    Xte = X_test_sc  if name != "Random Forest" else X_test.values

    model.fit(Xtr, y_train_bal)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    cv_auc = cross_val_score(model, Xtr, y_train_bal,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_scores[name] = cv_auc

    results[name] = {
        "Accuracy" : accuracy_score(y_test, y_pred),
        "F1"       : f1_score(y_test, y_pred, pos_label=1),
        "AUC"      : roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, pos_label=1),
        "Recall"   : recall_score(y_test, y_pred, pos_label=1),
        "CV_AUC"   : f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
        "y_prob"   : y_prob,
    }
    print(f"  {name:<22}  Acc={results[name]['Accuracy']:.4f}  "
          f"F1={results[name]['F1']:.4f}  AUC={results[name]['AUC']:.4f}")




[4] Baselines...
  KNN                     Acc=0.9561  F1=0.9383  AUC=0.9836
  SVM                     Acc=0.9825  F1=0.9762  AUC=0.9947
  Logistic Regression     Acc=0.9737  F1=0.9639  AUC=0.9944
  Random Forest           Acc=0.9737  F1=0.9630  AUC=0.9985


In [ ]:

#  ÉTAPE 5 — CATBOOST BASELINE (sans Optuna)

print("\n[5] CatBoost baseline (sans Optuna)...")

cat_base = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=RANDOM_STATE, verbose=0, eval_metric='AUC'
)
cat_base.fit(
    X_train_bal, y_train_bal,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=False
)

y_pred_cb = cat_base.predict(X_test)
y_prob_cb = cat_base.predict_proba(X_test)[:, 1]
cv_cb     = cross_val_score(cat_base, X_train_bal, y_train_bal,
                            cv=cv, scoring='roc_auc', n_jobs=-1)
cv_scores["CatBoost (baseline)"] = cv_cb

results["CatBoost (baseline)"] = {
    "Accuracy" : accuracy_score(y_test, y_pred_cb),
    "F1"       : f1_score(y_test, y_pred_cb, pos_label=1),
    "AUC"      : roc_auc_score(y_test, y_prob_cb),
    "Precision": precision_score(y_test, y_pred_cb, pos_label=1),
    "Recall"   : recall_score(y_test, y_pred_cb, pos_label=1),
    "CV_AUC"   : f"{cv_cb.mean():.4f} ± {cv_cb.std():.4f}",
    "y_prob"   : y_prob_cb,
}
print(f"  CatBoost baseline      Acc={results['CatBoost (baseline)']['Accuracy']:.4f}  "
      f"F1={results['CatBoost (baseline)']['F1']:.4f}  "
      f"AUC={results['CatBoost (baseline)']['AUC']:.4f}")



[5] CatBoost baseline (sans Optuna)...
  CatBoost baseline      Acc=0.9825  F1=0.9756  AUC=0.9993


In [ ]:

#  ÉTAPE 6 — OPTUNA (CatBoost + TPE, 150 trials)

print(f"\n[6] Optuna — {N_TRIALS} trials (TPE Sampler)...")

def objective(trial):
    params = {
        "iterations"          : trial.suggest_int("iterations", 200, 2000),
        "learning_rate"       : trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        "depth"               : trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg"         : trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature" : trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count"        : trial.suggest_int("border_count", 32, 255),
        "min_data_in_leaf"    : trial.suggest_int("min_data_in_leaf", 1, 50),
        "random_seed"         : RANDOM_STATE,
        "verbose"             : 0,
        "eval_metric"         : "AUC",
    }
    model  = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train_bal, y_train_bal,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\n  Meilleur AUC (CV) : {study.best_value:.6f}")
print("  Meilleurs hyperparamètres :")
for k, v in study.best_params.items():
    print(f"    {k:<28} = {v}")

# ── Figure 4 : Convergence Optuna 
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
vals = [t.value for t in study.trials if t.value is not None]
best = np.maximum.accumulate(vals)

axes[0].plot(vals, alpha=0.35, color='#2E75B6', lw=0.9, label='Trial AUC')
axes[0].plot(best, color='#E74C3C', lw=2.5, label=f'Best AUC = {study.best_value:.4f}')
axes[0].axhline(study.best_value, color='#E67E22', ls='--', lw=1.5)
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('AUC-ROC (CV 5-fold)')
axes[0].set_title('Convergence Optuna', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

imp = optuna.importance.get_param_importances(study)
imp_sorted = sorted(imp.items(), key=lambda x: x[1])
axes[1].barh([p[0] for p in imp_sorted], [p[1] for p in imp_sorted],
             color='#2E75B6', edgecolor='white')
axes[1].set_xlabel('Importance relative')
axes[1].set_title("Importance Hyperparamètres (Optuna)", fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig4_optuna.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n  ✓ fig4_optuna.png")


[6] Optuna — 150 trials (TPE Sampler)...


  0%|          | 0/150 [00:00<?, ?it/s]


  Meilleur AUC (CV) : 0.997599
  Meilleurs hyperparamètres :
    iterations                   = 235
    learning_rate                = 0.18013098499967797
    depth                        = 3
    l2_leaf_reg                  = 8.295581876010235
    bagging_temperature          = 0.3572843717119688
    border_count                 = 52
    min_data_in_leaf             = 42

  ✓ fig4_optuna.png


In [ ]:
# ── Entraînement modèle final 
best_params = {**study.best_params, "random_seed": RANDOM_STATE, "verbose": 0}
cat_opt = CatBoostClassifier(**best_params)
cat_opt.fit(
    X_train_bal, y_train_bal,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=False
)
cat_opt.save_model(f'{MODELS_DIR}/catboost_optuna.cbm')

y_pred_opt = cat_opt.predict(X_test)
y_prob_opt = cat_opt.predict_proba(X_test)[:, 1]
cv_opt     = cross_val_score(cat_opt, X_train_bal, y_train_bal,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
cv_scores["CatBoost + Optuna"] = cv_opt

results["CatBoost + Optuna"] = {
    "Accuracy" : accuracy_score(y_test, y_pred_opt),
    "F1"       : f1_score(y_test, y_pred_opt, pos_label=1),
    "AUC"      : roc_auc_score(y_test, y_prob_opt),
    "Precision": precision_score(y_test, y_pred_opt, pos_label=1),
    "Recall"   : recall_score(y_test, y_pred_opt, pos_label=1),
    "CV_AUC"   : f"{cv_opt.mean():.4f} ± {cv_opt.std():.4f}",
    "y_prob"   : y_prob_opt,
}
print(f"  CatBoost + Optuna      Acc={results['CatBoost + Optuna']['Accuracy']:.4f}  "
      f"F1={results['CatBoost + Optuna']['F1']:.4f}  "
      f"AUC={results['CatBoost + Optuna']['AUC']:.4f}")



  CatBoost + Optuna      Acc=0.9825  F1=0.9756  AUC=0.9954


In [ ]:
#  ÉTAPE 7 — SHAP (5 visualisations)
print("\n[7] SHAP — TreeExplainer...")

explainer   = shap.TreeExplainer(cat_opt)
shap_values = explainer.shap_values(X_test)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
feat_names = list(X_test.columns)

print(f"  SHAP values shape : {sv.shape}")

# ── Plot 1 : Summary Beeswarm 
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_test, plot_type="dot", max_display=20, show=False)
plt.title("SHAP Summary Plot — CatBoost + Optuna",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig5_shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig5_shap_summary.png")

# ── Plot 2 : Bar importance 
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_test, plot_type="bar", max_display=20, show=False)
plt.title("SHAP Feature Importance (Mean |SHAP|)",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig6_shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig6_shap_bar.png")

# ── Plot 3 : Waterfall (1 patient malin) 
malin_idx = np.where(y_test.values == 1)[0][0]
base_val  = (explainer.expected_value[1]
             if isinstance(explainer.expected_value, list)
             else explainer.expected_value)

plt.figure(figsize=(10, 7))
shap.waterfall_plot(
    shap.Explanation(
        values       = sv[malin_idx],
        base_values  = base_val,
        data         = X_test.iloc[malin_idx].values,
        feature_names= feat_names
    ),
    max_display=15, show=False
)
plt.title(f"SHAP Waterfall — Patient #{malin_idx} (Malin)",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig7_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig7_shap_waterfall.png")

# ── Plot 4 : Force plot (HTML interactif) 
force = shap.force_plot(base_val, sv[malin_idx],
                        X_test.iloc[malin_idx], show=False)
shap.save_html(f'{FIGURES_DIR}/fig8_shap_force.html', force)
print("  ✓ fig8_shap_force.html")

# ── Plot 5 : Dependence (worst radius) 
idx_wr = feat_names.index('worst radius') if 'worst radius' in feat_names else 0
plt.figure(figsize=(10, 6))
shap.dependence_plot(idx_wr, sv, X_test,
                     interaction_index="auto", show=False)
plt.title("SHAP Dependence — worst radius",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig9_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig9_shap_dependence.png")

# Top 5 features
mean_shap = np.abs(sv).mean(axis=0)
top5 = sorted(zip(feat_names, mean_shap), key=lambda x: -x[1])[:5]
print("\n  Top 5 features SHAP :")
for i, (f, v) in enumerate(top5, 1):
    print(f"    {i}. {f:<35} {v:.4f}")




[7] SHAP — TreeExplainer...
  SHAP values shape : (114, 30)
  ✓ fig5_shap_summary.png
  ✓ fig6_shap_bar.png
  ✓ fig7_shap_waterfall.png
  ✓ fig8_shap_force.html
  ✓ fig9_shap_dependence.png

  Top 5 features SHAP :
    1. worst texture                       1.0158
    2. worst concave points                0.9521
    3. area error                          0.9276
    4. worst radius                        0.7237
    5. worst perimeter                     0.5919


In [ ]:

#  ÉTAPE 8 — FIGURES ÉVALUATION
print("\n[8] Figures évaluation...")

COLORS = {
    "KNN"                : "#95A5A6",
    "SVM"                : "#8E44AD",
    "Logistic Regression": "#2E75B6",
    "Random Forest"      : "#E67E22",
    "CatBoost (baseline)": "#5D6D7E",
    "CatBoost + Optuna"  : "#E74C3C",
}


[8] Figures évaluation...


In [ ]:
# ── Figure 10 : Courbes ROC 
plt.figure(figsize=(9, 8))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    lw = 3 if name == "CatBoost + Optuna" else 1.5
    ls = '-' if name == "CatBoost + Optuna" else '--'
    plt.plot(fpr, tpr, lw=lw, ls=ls,
             color=COLORS.get(name, '#000'),
             label=f"{name}  (AUC={res['AUC']:.4f})")
plt.plot([0,1],[0,1],'k:',lw=1)
plt.xlabel('Taux Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux Vrais Positifs (TPR)', fontsize=12)
plt.title('Courbes ROC — Tous Modèles', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig10_roc.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig10_roc.png")

# ── Figure 11 : Precision-Recall 
plt.figure(figsize=(9, 7))
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ap  = average_precision_score(y_test, res['y_prob'])
    lw  = 3 if name == "CatBoost + Optuna" else 1.5
    plt.plot(rec, prec, lw=lw,
             color=COLORS.get(name, '#000'),
             label=f"{name}  (AP={ap:.4f})")
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Courbes Precision-Recall', fontsize=13, fontweight='bold')
plt.legend(loc='lower left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig11_precision_recall.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig11_precision_recall.png")

# ── Figure 12 : Confusion Matrix 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm      = confusion_matrix(y_test, y_pred_opt)
cm_norm = confusion_matrix(y_test, y_pred_opt, normalize='true')
for ax, mat, title in zip(axes,
                           [cm, cm_norm],
                           ['Matrice de Confusion', 'Matrice Normalisée']):
    disp = ConfusionMatrixDisplay(mat, display_labels=['Bénin','Malin'])
    disp.plot(ax=ax, cmap='Blues', colorbar=(mat is cm_norm))
    ax.set_title(f'{title}\nCatBoost + Optuna', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig12_confusion.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig12_confusion.png")

# ── Figure 13 : Learning Curve 
    cat_opt, X_train_bal, y_train_bal,
    cv=cv, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)
plt.figure(figsize=(9, 6))
plt.fill_between(tr_sz, tr_sc.mean(1)-tr_sc.std(1),
                 tr_sc.mean(1)+tr_sc.std(1), alpha=0.15, color='#2E75B6')
plt.fill_between(tr_sz, val_sc.mean(1)-val_sc.std(1),
                 val_sc.mean(1)+val_sc.std(1), alpha=0.15, color='#E74C3C')
plt.plot(tr_sz, tr_sc.mean(1),  'o-', color='#2E75B6', lw=2, label='Train AUC')
plt.plot(tr_sz, val_sc.mean(1), 's-', color='#E74C3C', lw=2, label='Validation AUC')
plt.xlabel("Taille d'entraînement", fontsize=12)
plt.ylabel('AUC-ROC', fontsize=12)
plt.title("Courbe d'Apprentissage — CatBoost + Optuna",
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig13_learning_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ fig13_learning_curve.png")



  ✓ fig10_roc.png
  ✓ fig11_precision_recall.png
  ✓ fig12_confusion.png
  ✓ fig13_learning_curve.png


In [62]:
# ══════════════════════════════════════════════════════════════════════════════
#  ÉTAPE 9 — TESTS STATISTIQUES
# ══════════════════════════════════════════════════════════════════════════════
print("\n[9] Tests statistiques...")

ref_scores = cv_scores["CatBoost + Optuna"]
print(f"\n  Test de Wilcoxon (CatBoost+Optuna vs baselines) :")
print(f"  {'Modèle':<25} {'p-value':>10}  {'Sig. p<0.05':>14}")
print("  " + "-" * 54)
for name, scores in cv_scores.items():
    if name == "CatBoost + Optuna":
        continue
    try:
        _, p = wilcoxon(ref_scores, scores)
        sig = "✅ OUI" if p < 0.05 else "❌ NON"
        print(f"  {name:<25} {p:>10.4f}  {sig:>14}")
    except Exception:
        print(f"  {name:<25} {'N/A':>10}")

all_cv = list(cv_scores.values())
if len(all_cv) >= 3:
    stat_f, p_f = friedmanchisquare(*all_cv)
    print(f"\n  Test de Friedman : χ²={stat_f:.3f}, p={p_f:.5f}")
    print(f"  → {'Différences significatives ✅' if p_f < 0.05 else 'Non significatif ❌'}")


# ══════════════════════════════════════════════════════════════════════════════
#  TABLEAU FINAL
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  RÉSULTATS FINAUX")
print("=" * 65)

df_res = pd.DataFrame({
    name: {
        "Accuracy" : f"{v['Accuracy']:.4f}",
        "F1-Score" : f"{v['F1']:.4f}",
        "AUC-ROC"  : f"{v['AUC']:.4f}",
        "Precision": f"{v['Precision']:.4f}",
        "Recall"   : f"{v['Recall']:.4f}",
        "CV AUC"   : v['CV_AUC'],
    }
    for name, v in results.items()
}).T

print(df_res.to_string())
df_res.to_csv('results.csv')

print(f"\n  Rapport de classification — CatBoost + Optuna :")
print(classification_report(y_test, y_pred_opt,
                             target_names=['Bénin (0)', 'Malin (1)']))

print("\n" + "=" * 65)
print(f"  AUC-ROC  : {results['CatBoost + Optuna']['AUC']:.4f}")
print(f"  Accuracy : {results['CatBoost + Optuna']['Accuracy']:.4f}")
print(f"  F1-Score : {results['CatBoost + Optuna']['F1']:.4f}")
print(f"  Recall   : {results['CatBoost + Optuna']['Recall']:.4f}  (classe Maligne)")
print(f"  CV AUC   : {results['CatBoost + Optuna']['CV_AUC']}")
print(f"\n  Figures  → ./{FIGURES_DIR}/  (13 fichiers)")
print(f"  Modèle   → ./{MODELS_DIR}/catboost_optuna.cbm")
print(f"  CSV      → ./results.csv")
print("=" * 65)
print("  ✅ TERMINÉ")
print("=" * 65)



[9] Tests statistiques...

  Test de Wilcoxon (CatBoost+Optuna vs baselines) :
  Modèle                       p-value     Sig. p<0.05
  ------------------------------------------------------
  KNN                           0.0625           ❌ NON
  SVM                           0.1250           ❌ NON
  Logistic Regression           0.2500           ❌ NON
  Random Forest                 0.0625           ❌ NON
  CatBoost (baseline)           0.1250           ❌ NON

  Test de Friedman : χ²=11.765, p=0.03816
  → Différences significatives ✅

  RÉSULTATS FINAUX
                    Accuracy F1-Score AUC-ROC Precision  Recall           CV AUC
KNN                   0.9561   0.9383  0.9836    0.9744  0.9048  0.9929 ± 0.0060
SVM                   0.9825   0.9762  0.9947    0.9762  0.9762  0.9951 ± 0.0043
Logistic Regression   0.9737   0.9639  0.9944    0.9756  0.9524  0.9963 ± 0.0029
Random Forest         0.9737   0.9630  0.9985    1.0000  0.9286  0.9938 ± 0.0029
CatBoost (baseline)   0.9825   0